In [ ]:
import logging

logging.basicConfig(
    filename="day2.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("internship_day2")
print("Logging configured -> day2.log")


Logging configured -> day2.log


In [ ]:
import pandas as pd

file_path = "data/Test.csv"

df = pd.read_csv(file_path)

logger.info("Loaded %s | shape=%s", file_path, df.shape)
print(df.shape)
print(df.head())

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

logger.info(
    "Data quality check | missing_total=%s | duplicates=%s",
    df.isnull().sum().sum(),
    df.duplicated().sum(),
)

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

logger.info("Removed %s duplicate row(s) | rows: %s -> %s", before - after, before, after)
print(f"Dropped {before - after} duplicate row(s). Rows remaining: {after}")

In [ ]:
for col in ["City", "Product", "Gender"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["City"] = df["City"].replace({"Ktm": "Kathmandu"})
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female"})

logger.info("Standardized categorical columns: City, Product, Gender")
print(df[["City", "Product", "Gender"]].nunique())

In [ ]:
# Numeric columns: fill with median
for col in ["Unit_Price", "Age"]:
    median_val = df[col].median()
    missing = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    if missing:
        logger.info("Filled %s missing value(s) in %s with median=%s", missing, col, median_val)

# Categorical columns: fill with an explicit placeholder rather than guessing
df["City"] = df["City"].fillna("Unknown")
df["Email"] = df["Email"].fillna("unknown@example.com")

logger.info("Handled missing values in Unit_Price, Age, City, Email")
print(df.isnull().sum())

In [ ]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", format="mixed")
invalid_dates = df["Order_Date"].isnull().sum()

logger.info("Converted Order_Date to datetime | invalid_dates=%s", invalid_dates)
print(f"Invalid/unparseable dates: {invalid_dates}")
print(df["Order_Date"].head())

In [ ]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "Unit_Price"]:
    lower, upper = iqr_bounds(df[col])
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    logger.info(
        "Capped %s outlier(s) in %s to range [%.2f, %.2f]", outliers, col, lower, upper
    )
    print(f"{col}: capped {outliers} outlier(s) to [{lower:.2f}, {upper:.2f}]")

In [ ]:
df["Total_Amount"] = df["Quantity"] * df["Unit_Price"]

logger.info("Added Total_Amount = Quantity * Unit_Price")
print(df[["Quantity", "Unit_Price", "Total_Amount"]].head())

In [ ]:
output_path = "data/Test_cleaned.csv"
df.to_csv(output_path, index=False)

logger.info("Saved cleaned data -> %s | shape=%s", output_path, df.shape)
print(f"Saved cleaned data to {output_path}")
print(df.shape)
df.head()

In [ ]:
logger.info("Day 2 completed successfully")